# Home Credit Risk Analysis

## Dataset: bureau_balance.csv

### Objetivo

Analizar la estructura, calidad y relaciones del dataset para comprender su papel dentro de la arquitectura Lakehouse y del modelo dimensional.

---

### Contexto del negocio

El dataset bureau_balance contiene el historial mensual de los créditos reportados por entidades financieras externas. 

Cada registro representa el estado de un crédito en un mes específico, permitiendo analizar su evolución temporal, comportamiento de pago y posibles situaciones de mora

Este dataset complementa la información contenida en bureau y constituye una fuente fundamental para el análisis histórico del riesgo crediticio. 

---

### Rol dentro de la arquitectura

application_train 
        │
        ▼
     bureau
        │
        ▼
  bureau_balance

# 1. Importación de librerías

Se cargan las librerías necesarias para el análisis exploratorio de datos.

In [3]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

# 2. Carga del Dataset

Se carga el dataset para realizar el análisis exploratorio y comprender su estructura, calidad y relevancia dentro del dominio de negocio.

In [ ]:
df = pd.read_csv("../../../data/raw/homeCredit/bureau_balance.csv")

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

Filas: 27299925
Columnas: 3


# 3. Tamaño del Dataset

## Resultados

| Métrica | Valor |
|----------|---------|
| Filas | 27,299,925 |
| Columnas | 3 |

## Interpretación

A pesar de contener solamente tres atributos, `bureau_balance.csv` es uno de los datasets mas voluminosos del ecosistema Home Credit, con mas de 27 millones de registros. 

Esta caracteristica indica que el dataset almacena información histórica con un alto nivel de detalle temporal, donde múltiples registros pueden estar asociados a un mismo crédito a lo largo del tiempo. 

La baja cantidad de columnas y el elevado volumen de filas sugieren una estructura orientada al seguimiento mensual del comportamiento crediticio, más que al almacenamiento de información descriptiva. 

## Información contenida 

El dataset registra principalmente: 

- Identificador deñ crédito histórico. 
- Referencia temporal mensual. 
- Estado del crédito en cada periodo. 

## Relevancia para el análisis 

Este nivel de granularidad permitirá reconstruir la evolución temporal de los créditos reportados por entidades financieras externas, facilitando el análisis de: 

- Historial de pagos. 
- Comportamiento de mora. 
- Evolución del estado del crédito. 
- Riesgo financiero acumulado. 

## Rol dentro del ecosistema Home Credit

Minetras que `bureau.csv` describe las caracteristicas generales de cada crédito histórico, `bureau_balance.csv` registra su comportamiento mensual. 

Por esta razón, ambos datasets se complementan para proporcionar una visión completa del historial crediticio de los cliente. 

# 4. Vista preliminar

Se inspeccionan los primeros registros para comprender la estructura general de los datos.

In [5]:
df.head()

,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C


## Muestra de Datos

Los registros observados contienen tres atributos principales:

| Columna | Descripción |
|----------|-------------|
| SK_ID_BUREAU | Identificador del crédito histórico |
| MONTHS_BALANCE | Referencia temporal en meses |
| STATUS | Estado del crédito durante el período |

---

## Observaciones Iniciales

La muestra evidencia que un mismo crédito puede aparecer múltiples veces dentro del dataset.

Por ejemplo:

SK_ID_BUREAU = 5715448

aparece asociado a varios valores de `MONTHS_BALANCE`:

- 0
- -1
- -2
- -3
- -4

Esto indica que el dataset almacena información histórica mensual y no únicamente una fotografía actual del crédito.

---

## Interpretación Temporal

La variable `MONTHS_BALANCE` parece representar la distancia temporal respecto al momento de referencia del análisis.

Ejemplo:

| MONTHS_BALANCE | Interpretación |
|----------------|---------------|
| 0 | Mes actual |
| -1 | Un mes antes |
| -2 | Dos meses antes |
| -3 | Tres meses antes |

Por lo tanto, cada registro representa el estado de un crédito en un momento específico del tiempo.

---

## Primera Hipótesis de Granularidad

A partir de la muestra observada se plantea la siguiente hipótesis:

1 registro = 1 crédito + 1 mes

Esta hipótesis será validada posteriormente mediante el análisis de claves y cardinalidad.

---

## Relación con bureau.csv

Mientras que `bureau.csv` almacena información general del crédito, `bureau_balance.csv` registra su evolución temporal.

Esto permite analizar:

- Historial de pagos.
- Comportamiento de mora.
- Cambios de estado del crédito.
- Evolución financiera a lo largo del tiempo.

Esta característica convierte a `bureau_balance` en una de las principales fuentes para la construcción de métricas históricas de riesgo crediticio.

# 5. Estructura General

Se analiza:

- Tipos de datos
- Cantidad de columnas
- Valores nulos
- Distribución general

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 27299925 entries, 0 to 27299924
Data columns (total 3 columns):
 #   Column          Dtype
---  ------          -----
 0   SK_ID_BUREAU    int64
 1   MONTHS_BALANCE  int64
 2   STATUS          str  
dtypes: int64(2), str(1)
memory usage: 650.9 MB


In [6]:
df.dtypes.value_counts()

int64    2
str      1
Name: count, dtype: int64

## Resumen Estructural

| Métrica | Valor |
|----------|---------|
| Registros | 27,299,925 |
| Columnas | 3 |
| Variables numéricas | 2 |
| Variables categóricas | 1 |
| Consumo de memoria | 650.9 MB |

---

## Tipos de Datos

| Columna | Tipo |
|----------|---------|
| SK_ID_BUREAU | int64 |
| MONTHS_BALANCE | int64 |
| STATUS | string |

---

## Interpretación de la Estructura

El dataset presenta una estructura altamente optimizada y especializada para el almacenamiento de información histórica.

A diferencia de otros datasets del dominio Home Credit, que contienen decenas o cientos de atributos, `bureau_balance.csv` concentra toda su información en únicamente tres columnas.

Esta simplicidad estructural facilita:

- Procesamiento distribuido.
- Agregaciones temporales.
- Consultas analíticas.
- Integración con modelos dimensionales.

---

## Componentes Principales

### SK_ID_BUREAU

Identifica el crédito histórico al que pertenece cada registro.

Esta columna permitirá relacionar el dataset con `bureau.csv`.

---

### MONTHS_BALANCE

Representa la dimensión temporal del historial.

Permite reconstruir la evolución mensual de cada crédito.

---

### STATUS

Describe la situación del crédito durante cada período observado.

Esta variable constituye el principal indicador de comportamiento crediticio dentro del dataset.

---

## Distribución de Tipos de Datos

| Tipo de dato | Cantidad |
|--------------|----------|
| int64 | 2 |
| string | 1 |

### Interpretación

El esquema del dataset presenta una estructura altamente simple y eficiente:

- Dos variables numéricas utilizadas para identificación y seguimiento temporal.
- Una variable categórica que describe el estado del crédito.

La baja diversidad de tipos de datos facilita los procesos de almacenamiento, particionamiento y transformación dentro de la arquitectura Lakehouse.

Además, reduce la complejidad de validación de esquemas durante las etapas Bronze y Silver.

---

## Consideraciones para Ingeniería de Datos

Aunque el esquema es reducido, el dataset posee más de 27 millones de registros, convirtiéndolo en uno de los conjuntos de datos más grandes del proyecto.

Esta característica implica:

- Necesidad de procesamiento distribuido mediante Spark.
- Particionamiento eficiente en Bronze y Silver.
- Optimización de transformaciones para evitar cuellos de botella.

El bajo número de columnas y la elevada cantidad de registros reflejan una estructura orientada a eventos históricos y seguimiento temporal.

# 6. Identificación de Claves

Se identifican:

- Clave primaria
- Claves foráneas
- Relaciones con otros datasets

In [7]:
len(df)

27299925

In [9]:
df["SK_ID_BUREAU"].nunique()

817395

In [17]:
df[["SK_ID_BUREAU", "MONTHS_BALANCE"]].duplicated().sum()

np.int64(0)

In [ ]:
df["SK_ID_BUREAU"].duplicated().sum

## Conclusiones sobre las Claves

A partir del análisis realizado se identificó que:

| Métrica | Valor |
|----------|---------|
| Registros totales | 27,299,925 |
| SK_ID_BUREAU únicos | 817,395 |
| Duplicados SK_ID_BUREAU | 26,482,530 |
| Duplicados (SK_ID_BUREAU, MONTHS_BALANCE) | 0 |

### Clave Primaria

La columna `SK_ID_BUREAU` no puede actuar como clave primaria individual debido a que un mismo crédito aparece múltiples veces dentro del dataset.

Sin embargo, la combinación:

`(SK_ID_BUREAU, MONTHS_BALANCE)`

identifica de manera única cada registro.

Por lo tanto, la clave primaria del dataset es una **clave compuesta** formada por:

### PK

(SK_ID_BUREAU, MONTHS_BALANCE)

---

### Clave Foránea

`SK_ID_BUREAU`

Esta columna permite relacionar cada registro mensual con el crédito histórico correspondiente almacenado en `bureau.csv`.

---

### Granularidad

Cada registro representa:

**1 crédito + 1 período mensual**

o equivalentemente:

**1 estado mensual de un crédito histórico**

---

### Cardinalidad

bureau (1)

↓

bureau_balance (N)

Un crédito puede presentar múltiples registros dentro de bureau_balance, uno por cada período mensual reportado.

---

### Interpretación de Negocio

El dataset no almacena información descriptiva del crédito, sino su evolución temporal.

Esto permite reconstruir el comportamiento histórico de cada obligación financiera y analizar cambios de estado, eventos de mora y evolución del riesgo a lo largo del tiempo.

## Conclusiones sobre las Claves

### Clave Primaria

La clave primaria del dataset está compuesta por: 

(SK_ID_BUREAU, MONTHS_BALANCE)

Esta combinación identifica de manera única cada registro y representa un estado específico de un crédito en un período mensual determinado. 

La validación realizada mostró que no existen duplicados para esta combinación de columnas. 

### Clave Foránea

SK_ID_BUREAU

Esta columna permite relacionar cada registro mensual con el credito histórico correspondiente almacenado en el dataset `bureau.csv`

### Cardinalidad

bureau (1)

↓

bureau_balance (N)

Un crédito histórico puede poseer múltiples registros mensuales dentro de bureau_balance.

### Granularidad

1 registro = 1 crédito + 1 mes

Cada fila representa el estado de un crédito en un período mensual específico.

### Interpretación de Negocio

El dataset almacena la evolución temporal de los créditos reportados por entidades financieras externas.

Esta estructura permite reconstruir el comportamiento histórico de cada obligación financiera y analizar eventos de mora, estados del crédito y patrones de pago a lo largo del tiempo.

# 7. Calidad de Datos

Se analiza la completitud de la información y posibles problemas de calidad.

In [21]:
null_percentage = (
    df.isnull()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

null_percentage.head(20)

df.isnull().sum()

SK_ID_BUREAU      0
MONTHS_BALANCE    0
STATUS            0
dtype: int64

In [20]:
high_nulls = null_percentage[null_percentage > 50]

medium_nulls = null_percentage[
    (null_percentage > 20)
    & (null_percentage <= 50)
]

low_nulls = null_percentage[
    (null_percentage > 0)
    & (null_percentage <= 20)
]

print("Columnas > 50%:", len(high_nulls))
print("Columnas entre 20%-50%:", len(medium_nulls))
print("Columnas < 20%:", len(low_nulls))

Columnas > 50%: 0
Columnas entre 20%-50%: 0
Columnas < 20%: 0


## Interpretación

El análisis de completitud muestra que el dataset presenta una calidad excepcional en términos de valores faltantes.

### Resultados

| Columna | % Nulos |
|----------|----------|
| SK_ID_BUREAU | 0% |
| MONTHS_BALANCE | 0% |
| STATUS | 0% |

### Conclusión

No se identificaron valores nulos en ninguna de las columnas del dataset.

Esto garantiza la disponibilidad completa de la información necesaria para reconstruir el comportamiento histórico de los créditos.

### Impacto en la Arquitectura Lakehouse

#### Bronze

Los datos podrán almacenarse sin requerir tratamientos adicionales de completitud.

#### Silver

No serán necesarias estrategias de imputación o manejo de valores faltantes.

Las validaciones de calidad se enfocarán principalmente en:

- Integridad referencial con bureau.
- Validez de los estados registrados.
- Consistencia temporal.

#### Gold y Diamond

La ausencia de valores faltantes favorece la construcción de indicadores históricos y métricas temporales sin necesidad de correcciones previas.

# 8. Detección de Registros Duplicados

In [22]:
df.duplicated().sum()

np.int64(0)

## Hallazgos

No se identificaron registros completamente duplicados dentro del dataset.

### Conclusión

La ausencia de registros duplicados confirma la consistencia estructural de la información histórica almacenada.

Este resultado complementa la validación realizada sobre la clave primaria compuesta:

(SK_ID_BUREAU, MONTHS_BALANCE)

y refuerza la confiabilidad del dataset para procesos analíticos y transformaciones posteriores.

### Implicaciones para la Arquitectura Lakehouse

#### Bronze

Los registros pueden almacenarse sin requerir procesos de deduplicación.

#### Silver

No será necesario implementar reglas de eliminación de duplicados para este dataset.

Las validaciones podrán enfocarse en:

- Integridad referencial con bureau.
- Consistencia temporal.
- Validez de los estados crediticios.

#### Gold y Diamond

La ausencia de duplicados garantiza la correcta construcción de métricas históricas y series temporales asociadas al comportamiento de los créditos.

# 9. Variables de Negocio Relevantes

Se analizan las variables con mayor relevancia para el dominio financiero.

In [24]:
important_cols = [
    "SK_ID_BUREAU",
    "MONTHS_BALANCE",
    "STATUS"
]

df[important_cols].head()

,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C


In [25]:
(
    df["STATUS"]
      .value_counts(normalize=True)
      .mul(100)
      .round(2)
)

STATUS
C    49.99
0    27.47
X    21.28
1     0.89
5     0.23
2     0.09
3     0.03
4     0.02
Name: proportion, dtype: float64

## Distribución de Estados del Crédito

| Estado | Porcentaje |
|----------|----------|
| C | 49.99% |
| 0 | 27.47% |
| X | 21.28% |
| 1 | 0.89% |
| 5 | 0.23% |
| 2 | 0.09% |
| 3 | 0.03% |
| 4 | 0.02% |

---

## Interpretación de Negocio

La variable `STATUS` representa el estado mensual del crédito y constituye el principal indicador de comportamiento financiero dentro del dataset.

### Significado de los Estados

| Estado | Interpretación |
|----------|----------|
| C | Crédito cerrado |
| X | Sin información para el período |
| 0 | Crédito al día |
| 1 | 1 mes de atraso |
| 2 | 2 meses de atraso |
| 3 | 3 meses de atraso |
| 4 | 4 meses de atraso |
| 5 | Mora severa o atraso prolongado |

---

## Hallazgos Principales

### Créditos Cerrados

El 49.99% de los registros corresponden a créditos cerrados (`C`).

Esto indica que aproximadamente la mitad del historial almacenado pertenece a obligaciones que ya finalizaron su ciclo de vida financiero.

---

### Créditos al Día

El 27.47% de los registros presentan estado `0`, lo que indica cumplimiento oportuno de las obligaciones.

La elevada proporción de registros sin mora sugiere un comportamiento financiero predominantemente estable.

---

### Registros sin Información

El estado `X` representa el 21.28% de los registros.

Estos casos requerirán análisis adicional durante las fases de transformación para determinar si corresponden a ausencia de actividad o situaciones específicas del proceso de reporte.

---

### Eventos de Mora

Los estados de atraso (1 a 5) representan conjuntamente menos del 1.3% de los registros.

Esto evidencia que los eventos de incumplimiento son relativamente poco frecuentes dentro del historial observado.

No obstante, debido a su relevancia para la evaluación del riesgo, estos registros poseen un alto valor analítico.

---

## Relevancia para el Modelo Analítico

La variable `STATUS` permitirá construir indicadores relacionados con:

- Número de meses en mora.
- Máxima mora histórica por crédito.
- Frecuencia de incumplimientos.
- Porcentaje de períodos al día.
- Riesgo histórico acumulado.
- Patrones de comportamiento financiero.

Estas métricas serán candidatas naturales para las capas Gold y Diamond de la arquitectura Lakehouse.

# 10. Variables Categóricas

In [26]:
categorical_columns = df.select_dtypes(include=["object", "string"])

categorical_columns.columns.tolist()

['STATUS']

In [27]:
for col in categorical_columns.columns:
    print(f"{col}: {df[col].nunique()}")

STATUS: 8


## Interpretación de la Variable STATUS

La variable `STATUS` describe el estado mensual de cada crédito reportado.

Con una cardinalidad de 8 categorías, constituye el principal atributo analítico del dataset.

### Estados identificados

| Estado | Interpretación |
|----------|----------|
| C | Crédito cerrado |
| X | Sin información para el período |
| 0 | Crédito al día |
| 1 | 1 mes de atraso |
| 2 | 2 meses de atraso |
| 3 | 3 meses de atraso |
| 4 | 4 meses de atraso |
| 5 | Mora severa o atraso prolongado |

---

### Clasificación Analítica

Las categorías pueden agruparse conceptualmente en tres grandes grupos:

#### Créditos Cerrados

- C

Representan obligaciones financieras finalizadas.

---

#### Créditos sin Información

- X

Corresponden a períodos donde no existe información reportada para el crédito.

---

#### Estados de Pago

- 0
- 1
- 2
- 3
- 4
- 5

Permiten medir distintos niveles de cumplimiento y mora.

---

## Relevancia para el Modelo Dimensional

La variable STATUS será utilizada para:

- Construir indicadores históricos de riesgo.
- Medir frecuencia de mora.
- Calcular máximos niveles de atraso.
- Analizar patrones de comportamiento financiero.
- Generar métricas agregadas para Gold y Diamond.

Debido a su baja cardinalidad y alto valor de negocio, esta variable constituye una excelente candidata para procesos de clasificación y enriquecimiento en las capas Silver e Intermediate.

# 11. Relaciones del Dataset

El dataset `bureau_balance.csv` forma parte del subsistema de historial crediticio externo y se encuentra directamente relacionado con `bureau.csv`.

Su función principal es complementar la información general de cada crédito mediante el almacenamiento de eventos históricos mensuales.

---

## Dataset Padre

### bureau.csv

Contiene la información general de cada crédito histórico reportado por entidades financieras externas.

La relación se realiza mediante:

SK_ID_BUREAU

---

## Dataset Hijo

No se identifican datasets descendientes directos dentro del conjunto Home Credit.

Por esta razón, bureau_balance representa el nivel más detallado de información dentro de la jerarquía de historial crediticio externo.

---

## Cardinalidad

bureau (1)

↓

bureau_balance (N)

Un crédito puede poseer múltiples registros mensuales asociados a distintos períodos históricos.

Cada registro de bureau_balance pertenece exclusivamente a un único crédito.

# 12. Conclusiones Técnicas

## Resumen Ejecutivo

El dataset `bureau_balance.csv` almacena el historial mensual de los créditos reportados por entidades financieras externas.

Su principal propósito es registrar la evolución temporal de cada crédito, permitiendo analizar comportamientos de pago, eventos de mora y estados históricos asociados a las obligaciones financieras de los clientes.

A diferencia de `bureau.csv`, que describe características generales del crédito, `bureau_balance.csv` registra eventos históricos ocurridos a lo largo del tiempo.

---

## Métricas Generales

| Métrica | Valor |
|----------|---------|
| Registros | 27,299,925 |
| Columnas | 3 |
| Clave Primaria | (SK_ID_BUREAU, MONTHS_BALANCE) |
| Clave Foránea | SK_ID_BUREAU |
| Duplicados | 0 |

---

## Calidad de Datos

El dataset presenta una estructura altamente consistente y compacta.

Las validaciones realizadas evidenciaron:

- Ausencia de registros duplicados.
- Clave primaria compuesta válida.
- Estructura homogénea.
- Bajo nivel de complejidad estructural.

La calidad observada facilita su integración dentro de los procesos analíticos y de transformación.

---

## Granularidad

Cada registro representa:

**1 crédito + 1 período mensual**

o equivalentemente:

**1 estado mensual de un crédito histórico**

Esta granularidad convierte a bureau_balance en el nivel más detallado de información dentro del subsistema de historial crediticio externo.

---

## Relaciones Identificadas

### Relación con bureau

bureau (1)

↓

bureau_balance (N)

Un crédito histórico puede presentar múltiples registros asociados a diferentes períodos mensuales.

---

### Jerarquía del Dominio

application_train

↓

bureau

↓

bureau_balance

| Dataset | Granularidad |
|----------|-------------|
| application_train | Cliente / Solicitud |
| bureau | Crédito |
| bureau_balance | Crédito / Mes |

---

## Variables de Negocio Relevantes

Las variables más importantes identificadas fueron:

- SK_ID_BUREAU
- MONTHS_BALANCE
- STATUS

La variable `STATUS` concentra la mayor parte del valor analítico del dataset y permite identificar el comportamiento histórico de los créditos.

---

## Distribución de Estados

| Estado | Participación |
|----------|-------------|
| C | 49.99% |
| 0 | 27.47% |
| X | 21.28% |
| 1 | 0.89% |
| 5 | 0.23% |
| 2 | 0.09% |
| 3 | 0.03% |
| 4 | 0.02% |

### Hallazgos

- Aproximadamente la mitad de los registros corresponden a créditos cerrados.
- Más de una cuarta parte de los registros representan créditos al día.
- Los eventos de mora severa representan una proporción reducida del historial observado.
- La variable STATUS permite reconstruir patrones históricos de riesgo financiero.

---

## Rol dentro del Modelo Dimensional

El dataset constituye un candidato natural para:

### FACT_BUREAU_MONTHLY_STATUS

Debido a que registra eventos históricos asociados a una dimensión temporal.

Su estructura permitirá construir indicadores relacionados con:

- Historial de mora.
- Máxima mora registrada.
- Meses en incumplimiento.
- Frecuencia de atraso.
- Comportamiento histórico del crédito.

---

## Rol dentro de la Arquitectura Lakehouse

bureau_balance.csv

↓ Bronze (ingesta cruda)

↓ Silver (validación y normalización)

↓ Intermediate (agregaciones temporales)

↓ Gold (indicadores históricos de riesgo)

↓ Diamond (convergencia corporativa)

---

## Recomendaciones para la Siguiente Fase

1. Integrar bureau y bureau_balance para construir métricas históricas por crédito.
2. Definir reglas Silver para la normalización de STATUS.
3. Crear indicadores derivados de mora y comportamiento de pago.
4. Diseñar agregaciones por cliente para la capa Intermediate.
5. Incorporar métricas históricas de riesgo en Gold y Diamond.

---

## Conclusión Final

bureau_balance constituye la fuente temporal más importante del subsistema de historial crediticio externo.

Su nivel de detalle mensual permite reconstruir el comportamiento histórico de los créditos y generar indicadores avanzados de riesgo financiero que complementarán la información descriptiva contenida en bureau y application_train.